In [1]:
from pathlib import Path

import requests

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent


In [9]:
print("hello")

hello


In [2]:
year = [2025,2024,2023,2022,2021,2020,2019,2018,2017,2016,2015]
url = "https://www.gov.br/mj/pt-br/assuntos/sua-seguranca/seguranca-publica/estatistica/download/dnsp-base-de-dados/bancovde-{y}.xlsx/@@download/file"
download_path = project_root / "data" / "xlsx_dowloads"

In [3]:
def download_xlsx(url: str, year: int, download_dir: str | Path) -> Path:
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)

    download_url = url.format(y=year)
    file_path = download_dir / f"pubseg-{year}.xlsx"
    response = requests.get(download_url, timeout=60)
    response.raise_for_status()
    file_path.write_bytes(response.content)

    return file_path

In [4]:
import pandas as pd
from sqlalchemy import (
    BigInteger,
    Boolean,
    Column,
    DateTime,
    Float,
    MetaData,
    String,
    Table,
    Text,
    create_engine,
    inspect,
)
from sqlalchemy.engine import URL
from sqlalchemy.schema import CreateSchema


def build_postgres_engine(host: str, port: int, database: str, user: str, password: str):
    return create_engine(
        URL.create(
            "postgresql+psycopg2",
            username=user,
            password=password,
            host=host,
            port=port,
            database=database,
        )
    )


def pandas_dtype_to_sqlalchemy(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return BigInteger
    if pd.api.types.is_float_dtype(dtype):
        return Float
    if pd.api.types.is_bool_dtype(dtype):
        return Boolean
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return DateTime
    return Text


def create_table_from_dataframe(
    dataframe: pd.DataFrame,
    host: str,
    port: int,
    database: str,
    user: str,
    password: str,
    schema_name: str,
    table_name: str,
) -> Table:
    if "id_tabela" not in dataframe.columns:
        raise KeyError("O dataframe precisa ter a coluna 'id_tabela'.")

    metadata = MetaData(schema=schema_name)
    columns = [Column("id", String, primary_key=True)]
    for column_name, dtype in dataframe.dtypes.items():
        columns.append(
            Column(
                str(column_name),
                pandas_dtype_to_sqlalchemy(dtype),
                nullable=bool(dataframe[column_name].isna().any()),
            )
        )

    table = Table(table_name, metadata, *columns)
    engine = build_postgres_engine(host, port, database, user, password)

    with engine.begin() as connection:
        if not inspect(connection).has_schema(schema_name):
            connection.execute(CreateSchema(schema_name))
        metadata.create_all(connection, tables=[table], checkfirst=True)

    return table

In [5]:
def upload_dataframe_to_postgres(
    dataframe: pd.DataFrame,
    host: str,
    port: int,
    database: str,
    user: str,
    password: str,
    schema_name: str,
    table_name: str,
) -> int:
    if "id_tabela" not in dataframe.columns:
        raise KeyError("O dataframe precisa ter a coluna 'id_tabela'.")
    if dataframe["id_tabela"].isna().any():
        raise ValueError("A coluna 'id_tabela' não pode ter valores nulos.")

    frame = dataframe.copy()
    id_series = frame["id_tabela"].astype("string").str.replace(r"\.0+$", "", regex=True)
    frame.insert(0, "id", id_series)

    engine = build_postgres_engine(host, port, database, user, password)
    with engine.begin() as connection:
        frame.to_sql(
            table_name,
            con=connection,
            schema=schema_name,
            if_exists="append",
            index=False,
            method="multi",
        )

    return len(frame)

In [6]:
from pathlib import Path

from dotenv import dotenv_values

env = dotenv_values(project_root / ".env")
host = "localhost"
port = int(env.get("POSTGRES_PORT", 5432))
database = env["POSTGRES_DB"]
user = env["POSTGRES_USER"]
password = env["POSTGRES_PASSWORD"]
schema_name = "public"
table_name = "pubseg"

from sqlalchemy import text


def drop_table(schema_name: str, table_name: str) -> None:
    engine = build_postgres_engine(host, port, database, user, password)

    with engine.begin() as connection:
        connection.execute(text(f'DROP TABLE IF EXISTS \"{schema_name}\".\"{table_name}\"'))

drop_table(schema_name, table_name)


In [7]:
def table_ajust(db: pd.DataFrame):

    year_series = pd.to_datetime(db["data_referencia"], errors="coerce").dt.year

    if year_series.isna().any():
        raise ValueError("The 'data_referencia' column must contain valid dates to generate the id_tabela.")
    
    row_id = pd.Series(range(1, len(db) + 1), index=db.index, dtype="int64")
    db["id_tabela"] = year_series.astype("Int64").astype("string") + row_id.astype("string")
    ordered_columns = ["id_tabela", *[column for column in db.columns if column != "id_tabela"]]
    db = db[ordered_columns]

    db["feminino"] = (
        pd.to_numeric(db["feminino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["masculino"] = (
        pd.to_numeric(db["masculino"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["nao_informado"] = (
        pd.to_numeric(db["nao_informado"], errors="coerce")  # garante numérico
        .fillna(0)                                     # NaN → 0
        .astype(int)
    )
    db["total_vitima"] = (
        pd.to_numeric(db["total_vitima"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    return db


Se não for a primeira vez rodando o script pode comentar essa parte

In [8]:
path = download_xlsx(url, year[0], download_path)
db = pd.read_excel(path)
db = table_ajust(db)
create_table_from_dataframe(
    db,
    host=host,
    port=port,
    database=database,
    user=user,
    password=password,
    schema_name=schema_name,
    table_name=table_name,
)

ChunkedEncodingError: ('Connection broken: IncompleteRead(1441792 bytes read, 26374080 more expected)', IncompleteRead(1441792 bytes read, 26374080 more expected))

In [ ]:

def dowload_full_db(url:str, year:list[int], download_dir: str | Path, 
                    schema_name:str, table_name:str):
    for y in year:
        try:
            print(f"Start dowload: {y} data")
            path = download_xlsx(url, y, download_dir)
            db = pd.read_excel(path)
            db = table_ajust(db)
            print(f"Succes on dowload: {y} data")

        except Exception as e:
            print(f"error during {y} data dowload: {e}")
            continue
        try:
            rows_inserted = upload_dataframe_to_postgres(
                db,
                host=host,
                port=port,
                database=database,
                user=user,
                password=password,
                schema_name=schema_name,
                table_name=table_name,
            )

            print(f"Lines inserted in {schema_name}.{table_name}: {rows_inserted}, from {y} data")

        except:
            raise


        

In [ ]:
dowload_full_db(url, year, download_path, schema_name, table_name)

In [ ]:
# year = [2022,2020,2016]
# dowload_full_db(url, year, download_path, schema_name, table_name)

In [ ]:
# year = ["segundo-semestre-de-2021", "primeiro-semestre-de-2021"]
# url = "https://www.gov.br/mdh/pt-br/acesso-a-informacao/dados-abertos/disque100/{y}"
# download_path = project_root / "data" / "csv_dowloads"

# schema_name = "public"
# table_name = "raw_disk100"

# path=download_xlsx(url, year[0], download_path)

# db= pd.read_csv(path, sep=";", encoding="latin1")

In [ ]:
# db.columns = (
#     db.columns
#     .str.strip()
#     .str.lower()
#     .str.replace(" ", "_", regex=False)
# )

# db["data_de_cadastro"] = pd.to_datetime(
#     db["data_de_cadastro"], format="%d/%m/%Y %H:%M", errors="coerce"
# )
# db["sl_quantidade_vitimas"] = pd.to_numeric(
#     db["sl_quantidade_vitimas"], errors="coerce"
# ).astype("Int64")
# db["id_tabela"] = db["data_de_cadastro"].dt.strftime("%Y%m%d%H%M%S")




In [ ]:
# create_table_from_dataframe(
#     db,
#     host=host,
#     port=port,
#     database=database,
#     user=user,
#     password=password,
#     schema_name=schema_name,
#     table_name=table_name,
# )

In [ ]:
# rows_inserted = upload_dataframe_to_postgres(
#                 db,
#                 host=host,
#                 port=port,
#                 database=database,
#                 user=user,
#                 password=password,
#                 schema_name=schema_name,
#                 table_name=table_name,
#             )

# print(f"Lines inserted in {schema_name}.{table_name}: {rows_inserted}, from {y} data")